# Retail Sales Data Analysis
A beginner-friendly walkthrough using a synthetic retail dataset. Review the README for assumptions and limitations.

## 1. Load libraries and data

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
data_path = ROOT / 'data' / 'retail_sales.csv'
df = pd.read_csv(data_path)
df.head()

## 2. Inspect and clean the data

In [ ]:
df.info()
df.isna().sum()

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['product'] = df['product'].astype('string').str.strip()
df['region'] = df['region'].fillna('Unknown')
df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
df['unit_price'] = pd.to_numeric(df['unit_price'], errors='coerce')
df['discount_pct'] = pd.to_numeric(df['discount_pct'], errors='coerce').fillna(0)
df['unit_cost'] = pd.to_numeric(df['unit_cost'], errors='coerce')
df = df.drop_duplicates().dropna(subset=['order_date','product','quantity','unit_price','unit_cost'])
df = df[df['quantity'] > 0].copy()
df['gross_sales'] = df['quantity'] * df['unit_price']
df['net_sales'] = df['gross_sales'] * (1 - df['discount_pct'])
df['total_cost'] = df['quantity'] * df['unit_cost']
df['profit_estimate'] = df['net_sales'] - df['total_cost']
df['month'] = df['order_date'].dt.to_period('M').astype(str)
df.head()

## 3. Summary metrics

In [ ]:
print(f'Rows after cleaning: {len(df):,}')
print(f'Net sales: ${df["net_sales"].sum():,.2f}')
print(f'Estimated profit: ${df["profit_estimate"].sum():,.2f}')
print(f'Average net sales per transaction line: ${df["net_sales"].mean():,.2f}')

## 4. Monthly sales trend

In [ ]:
monthly = df.groupby('month', as_index=False)['net_sales'].sum()
plt.figure(figsize=(10,5))
plt.plot(monthly['month'], monthly['net_sales'], marker='o')
plt.title('Monthly Net Sales')
plt.xlabel('Month')
plt.ylabel('Net sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 5. Product and regional performance

In [ ]:
top_products = df.groupby('product', as_index=False)['net_sales'].sum().sort_values('net_sales', ascending=False)
display(top_products)
top_products.plot(kind='barh', x='product', y='net_sales', legend=False, figsize=(9,5), title='Net Sales by Product')
plt.tight_layout()
plt.show()

In [ ]:
region_sales = df.groupby('region', as_index=False)['net_sales'].sum().sort_values('net_sales', ascending=False)
display(region_sales)
region_sales.plot(kind='bar', x='region', y='net_sales', legend=False, figsize=(8,5), title='Net Sales by Region')
plt.tight_layout()
plt.show()

## 6. Your own interpretation
Write 3–5 sentences here after running the notebook: Which month was strongest? Which products performed best? Did one region stand out? What are the limitations of the synthetic data? Avoid inventing business conclusions not supported by your results.